In [5]:
import tqdm
import torch
import glob
import networkx as nx
import numpy as np

# Gather all graph file paths
dataset = "IMDB"
# dataset="Imprisonment-IT"
train_paths = glob.glob(f"../data/datasets/{dataset}/train/interim/*.pt")
val_paths = glob.glob(f"../data/datasets/{dataset}/validation/interim/*.pt")
test_paths = glob.glob(f"../data/datasets/{dataset}/test/interim/*.pt")

all_paths = train_paths + val_paths + test_paths

metrics = {
    "num_graphs": 0,
    "node_counts": [],
    "edge_counts": [],
    "avg_in_degrees": [],
    "avg_out_degrees": [],
    "clustering_coeffs": [],
    "densities": [],
    "diameters": [],
    "avg_shortest_paths": [],
    "connected_components": [],
    "assortativities": []
}

for path in tqdm.tqdm(all_paths):
    doc_name, label, graph = torch.load(path)
    if not isinstance(graph, nx.MultiDiGraph):
        continue

    metrics["num_graphs"] += 1
    n_nodes = graph.number_of_nodes()
    n_edges = graph.number_of_edges()

    metrics["node_counts"].append(n_nodes)
    metrics["edge_counts"].append(n_edges)

    in_degrees = [deg for _, deg in graph.in_degree()]
    out_degrees = [deg for _, deg in graph.out_degree()]
    if in_degrees:
        metrics["avg_in_degrees"].append(np.mean(in_degrees))
    if out_degrees:
        metrics["avg_out_degrees"].append(np.mean(out_degrees))

    # Convert to undirected simple graph for clustering coefficient
    undirected_simple = nx.Graph()
    for u, v in graph.edges():
        undirected_simple.add_edge(u, v)

    if undirected_simple.number_of_nodes() > 1:
        try:
            cc = nx.average_clustering(undirected_simple)
            metrics["clustering_coeffs"].append(cc)
        except Exception:
            pass

    metrics["densities"].append(nx.density(graph))
    metrics["connected_components"].append(nx.number_weakly_connected_components(graph))

    # Convert to DiGraph (no multiedges) for shortest path & diameter
    simple_digraph = nx.DiGraph()
    simple_digraph.add_edges_from(graph.edges())

    try:
        if nx.is_weakly_connected(simple_digraph):
            sp = nx.average_shortest_path_length(simple_digraph)
            dia = nx.diameter(simple_digraph.to_undirected())
            metrics["avg_shortest_paths"].append(sp)
            metrics["diameters"].append(dia)
    except Exception:
        pass

    try:
        assortativity = nx.degree_assortativity_coefficient(graph)
        if not np.isnan(assortativity):
            metrics["assortativities"].append(assortativity)
    except Exception:
        pass

# Median ± Std summary
def summarize(values):
    if values:
        arr = np.array(values)
        return f"{np.median(arr):.4f} ± {np.std(arr):.4f}"
    return "N/A"

summary = {
    "total_graphs": metrics["num_graphs"],
    "nodes per graph (median ± std)": summarize(metrics["node_counts"]),
    "edges per graph (median ± std)": summarize(metrics["edge_counts"]),
    "avg in-degree (median ± std)": summarize(metrics["avg_in_degrees"]),
    "avg out-degree (median ± std)": summarize(metrics["avg_out_degrees"]),
    "avg clustering (median ± std)": summarize(metrics["clustering_coeffs"]),
    "density (median ± std)": summarize(metrics["densities"]),
    "diameter (median ± std)": summarize(metrics["diameters"]),
    "avg shortest path (median ± std)": summarize(metrics["avg_shortest_paths"]),
    "weakly connected components (median ± std)": summarize(metrics["connected_components"]),
    "assortativity (median ± std)": summarize(metrics["assortativities"]),
}

print("\n=== MultiDiGraph Metrics Summary ===")
for key, value in summary.items():
    print(f"{key}: {value}")

  0%|          | 0/50000 [00:00<?, ?it/s]/tmp/ipykernel_10322/3671102334.py:31: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  doc_name, label, graph = torch.load(path)
100%|


=== MultiDiGraph Metrics Summary ===
total_graphs: 50000
nodes per graph (median ± std): 115.0000 ± 79.7852
edges per graph (median ± std): 341.0000 ± 336.6609
avg in-degree (median ± std): 2.9767 ± 0.6058
avg out-degree (median ± std): 2.9767 ± 0.6058
avg clustering (median ± std): 0.3769 ± 0.0437
density (median ± std): 0.0255 ± 0.0403
diameter (median ± std): 7.0000 ± 1.0979
avg shortest path (median ± std): 4.6342 ± 0.7588
weakly connected components (median ± std): 1.0000 ± 0.4415
assortativity (median ± std): -0.0214 ± 0.0851
